# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE)

## Notebook P2-06: MCD-rPPG Ensemble rPPG Extraction (FullHDwebcam)

### What this notebook does

This notebook extracts rPPG signals from the **1200 FullHDwebcam frontal recordings** of the MCD-rPPG dataset using a **three-algorithm ensemble** (POS, CHROM, GREEN), producing a single higher-quality rPPG signal per recording written to a new `mcd_rppg_ensemble.h5`.

### Why ensemble instead of a single algorithm

No single traditional rPPG algorithm dominates across all conditions. Each method carries different assumptions about skin optics and illumination, and their failure modes are complementary:

| Algorithm | Core assumption | Strength | Failure mode|
|-----------|-----------------|----------|-------------|
| **POS** | Projects chrominance signal onto a plane orthogonal to the skin tone vector | Robust under illumination variation. Good SNR in naturalistic settings| Degrades when sking-tone assumption is violated (very dark or very light tones)|
| **CHROM** | Models skin as a linear mixture of specular + diffuse reflectance | Strong specular glare rejection. Reliable on lighter skin | More sensitive to motion artefacts. Documented MAE degradation on darker skin tones |
| **GREEN** | The pulsatile signal is strongest in the green channel | Computationally trivial. Interpretable baseline | No artefact rejection. Noise dominated in motion or poor lighting | 


### Ensembling strategy 

Signal level fusion after per signal normalisation:

```
POS signal  [T]  ──┐
CHROM signal [T] ──┼─> z-score normalise each -> weighted sum -> ensemble rPPG [T]
GREEN signal [T] ──┘
```

**Weights** are derived per-recording from spectral SQI (peak-to-band power ratio in 0.7–3.5 Hz):

```
w_i = SQI_i / Σ SQI_j        (soft quality-proportional weighting)
```

Weights sum to 1. A floor (`RPPG_SQI_MIN_FLOOR`) prevents division-by-zero if all algorithms produce near-zero SQI. Per-algorithm SQI scores and weights are stored in HDF5 for post-hoc skin-tone fairness analysis.

### Reference strategy (identical to NB_P2_04)

- **ECG** (~30 s, 500 Hz): HRV features (RMSSD, SDNN, pNN50)
- **PPG** (~180 s, ~100 Hz): `hr_continuous` and `reference_signal`  
- `reference_type = "ECG_hrv_PPG_continuous"`
- 

### Pipeline

```
{subject_id}_FullHDwebcam_{state}.avi  (29.90 FPS, frontal)
    │
    ├─ 1  BGR -> RGB
    ├─ 2  MediaPipe Face Mesh -> ROI masks (forehead, left cheek, right cheek)
    └─ 3  Spatial mean RGB per ROI -> roi_rgb [T, 3_ROIs, 3_ch]
               │
               ├─ 4a  POS algorithm   -> rppg_pos   [T]
               ├─ 4b  CHROM algorithm -> rppg_chrom [T]
               └─ 4c  GREEN mean      -> rppg_green [T]
                         │
                         ├─ 5  Bandpass 0.7–3.5 Hz + z-score (all three)
                         ├─ 6  Spectral SQI: peak/band power ratio (all three)
                         └─ 7  Quality-weighted ensemble → rppg_ensemble [T]

ECG (500 Hz JSON) -> bandpass -> R-peaks -> HRV + ECG SQI
PPG (100 Hz .PW) -> bandpass -> peaks -> hr_continuous + reference_signal

 8  Gate: ECG SQI ≥ 0.5 AND ensemble SQI ≥ 0.07 
 9  Write to mcd_rppg_ensemble.h5  
```

### SQI gate rationale

```
Initial gate : RPPG_SQI_THRESHOLD = 0.10
Final gate: RPPG_SQI_THRESHOLD = 0.07
```

The spectral SQI used here (peak-to-band power ratio, ±3 bins around dominant frequency) differs fundamentally from NB_P2_04's autocorrelation SQI. For a real cardiac signal of ~180 s at 29.90 FPS, the FFT has 0.0056 Hz/bin. Normal HR variability broadens the spectral peak across ~20-30 bins, so a clean signal naturally yields SQI in the 0.10-0.35 rather than -> 1.0. Post-exercise recordings compund this: residual motion artefact and elevated HR further broaden the peak, pushing genuine signals toward 0.07-0.10.

Calibration on 20 subjects showed a natural gap between passing (0.07-0.321) and failing (0.037-0.069) recordings, with no dense cluster of bordeline cases at the 0.07 boundary. Lowering from 0.10 to 0.07 recovered post-exercise recordings with physiologically plausible HR (75-103 BPM), good ECG SQI (0.87-0.96), and 0% missed faces frames - indicating genuine signal degradation from exercise physiology rather than  sensor or detection failure.




## 1. Enviroment Setup

In [17]:
import subprocess, sys

packages = [
    "neurokit2",
    "mediapipe",
    "h5py",
    "opencv-python",
    "scipy",
    "numpy",
    "pandas",
    "matplotlib"
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("All packages confirmed.")

All packages confirmed.


## 2. Imports

In [18]:
import os, warnings, cv2, h5py, urllib.request, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import neurokit2 as nk
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from scipy.signal import butter, filtfilt
warnings.filterwarnings("ignore")

# Plot style 
plt.rcParams.update({
    "figure.facecolor" : "#0f0f0f",
    "axes.facecolor" : "#1a1a2e",
    "axes.edgecolor" : "#444444",
    "axes.labelcolor" : "#e0e0e0",
    "xtick.color" : "#e0e0e0",
    "ytick.color" : "#e0e0e0",
    "text.color" : "#e0e0e0",
    "grid.color"  : "#2a2a3e",
    "grid.linestyle" : "--",
    "grid.alpha" : 0.5,
    "figure.dpi" : 110,
})

# Paths 
MCD_DIR = "F:/MCD-rppg/MCD_rPPG_dataset"
OUTPUT_DIR = "E:/QualityPhys"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Data Exploration


In [19]:
db_path = os.path.join(MCD_DIR, "db.csv")
df_db = pd.read_csv(db_path)

print("=" * 60)
print("1. db.csv - FullHdwebcab rows")
print("=" * 60)
df_front = df_db[df_db["camera"] == "FullHDwebcam"]
print(f"Total db.csv rows : {len(df_db)}")
print(f"FullHDwebcam rows : {len(df_front)}")
print(f"Subjects : {df_front['patient_id'].nunique()}")
print(f"States : {sorted(df_front['step'].unique())}")
print()

print("=" * 60)
print("2. FPS from actual video headers - all cameras")
print("=" * 60)
camera_fps = {}
for cam in ["FullHDwebcam", "USBVideo", "IriunWebcam"]:
    rows = df_db[df_db["camera"] == cam]
    if len(rows) == 0:
        continue
    sample_path = os.path.join(MCD_DIR, rows.iloc[0]["video"])
    if not os.path.exists(sample_path):
        print(f"{cam:<16} sample video not found: {sample_path}")
        continue
    cap = cv2.VideoCapture(sample_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    view = rows.iloc[0]["view"]
    ok = "OK frontal" if view == "front" else "X profile - excluded"
    camera_fps[cam] = fps
    print(f"{cam:<16} FPS={fps:.4f} {w}x{h} frames={n} view={view} {ok}")
print()

print("=" * 60)
print("3. FPS distribution across all FullHDwebcam videos")
print("=" * 60)
fps_all = []
for _, row in df_front.iterrows():
    vpath = os.path.join(MCD_DIR, row["video"])
    if os.path.exists(vpath):
        cap = cv2.VideoCapture(vpath)
        fps_v = cap.get(cv2.CAP_PROP_FPS)
        cap.release()
        if fps_v > 0:
            fps_all.append(fps_v)
if fps_all:
    fps_arr = np.array(fps_all)
    print(f"Videos checked : {len(fps_arr)}")
    print(f"FPS mean={fps_arr.mean():.4f} std={fps_arr.std():.5f} min={fps_arr.min():.4f} max={fps_arr.max():.4f}")
    n_inconsistent = (np.abs(fps_arr - fps_arr.mean()) > 0.1).sum()
    print(f"Videos with FPS > 0.1 from mean: {n_inconsistent}")
print()

print("=" * 60)
print("4. ECG format (JSON)")
print("=" * 60)
ecg_rel = df_db[df_db["ecg"].notna()].iloc[0]["ecg"]
ecg_path = os.path.join(MCD_DIR, ecg_rel)
with open(ecg_path) as file:
    ecg_j = json.load(file)
ecg_fs_actual = float(ecg_j["frequency"])
leads_found = [lead["title"] for lead in ecg_j["data"]]
n_samp_ecg = len(ecg_j["data"][0]["values"])
ecg_duration = n_samp_ecg / ecg_fs_actual
print(f"frequency={ecg_fs_actual} Hz leads={leads_found} samples={n_samp_ecg} ({ecg_duration:.0f} s)")
print()

print("=" * 60)
print("5. PPG format (.PW)")
print("=" * 60)
ppg_rel = df_db[df_db["ppg"].notna()].iloc[0]["ppg"]
ppg_path = os.path.join(MCD_DIR, ppg_rel)
df_pw = pd.read_csv(ppg_path, sep=r"\s+", header=None, dtype=str)
ppg_raw = df_pw.iloc[:, 0].astype(np.float32).values
ts_str = df_pw.iloc[:, 1] + " " + df_pw.iloc[:, 2]
ts_dt = pd.to_datetime(ts_str, format="mixed")
ts_s = (ts_dt - ts_dt.iloc[0]).dt.total_seconds().values
ppg_fs_actual = 1.0 / np.diff(ts_s).mean()
ppg_duration = ts_s[-1]
print(f"fs≈{ppg_fs_actual:.1f} Hz samples={len(ppg_raw)} duration={ppg_duration:.0f} s amplitude range: {ppg_raw.min():.0f}-{ppg_raw.max():.0f}")
print()

print("=" * 60)
print("SUMMARY:")
print("=" * 60)
for cam, fps in camera_fps.items():
    print(f"{cam} FPS = {fps:.4f}")
print(f"ECG_FS = {ecg_fs_actual:.0f} Hz")
print(f"ECG_DURATION_S = {ecg_duration:.0f} s")
print(f"PPG_FS ≈ {ppg_fs_actual:.0f} Hz")


1. db.csv - FullHdwebcab rows
Total db.csv rows : 3600
FullHDwebcam rows : 1200
Subjects : 600
States : ['after', 'before']

2. FPS from actual video headers - all cameras
FullHDwebcam     FPS=29.9000 640x480 frames=0 view=front OK frontal
USBVideo         FPS=30.0000 640x480 frames=0 view=left X profile - excluded
IriunWebcam      FPS=24.0000 640x480 frames=0 view=right X profile - excluded

3. FPS distribution across all FullHDwebcam videos
Videos checked : 1200
FPS mean=28.1887 std=2.68248 min=24.0000 max=30.0000
Videos with FPS > 0.1 from mean: 1200

4. ECG format (JSON)
frequency=500.0 Hz leads=['I', 'II', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'III', 'aVR', 'aVL', 'aVF'] samples=14900 (30 s)

5. PPG format (.PW)
fs≈100.0 Hz samples=18051 duration=180 s amplitude range: 0-236

SUMMARY:
FullHDwebcam FPS = 29.9000
USBVideo FPS = 30.0000
IriunWebcam FPS = 24.0000
ECG_FS = 500 Hz
ECG_DURATION_S = 30 s
PPG_FS ≈ 100 Hz


## 4. Configuration


In [20]:
DATASET_NAME = "mcd_rppg_ensemble"
HDF5_PATH = os.path.join(OUTPUT_DIR, "mcd_rppg_ensemble.h5")
MP_MODEL_PATH = os.path.join(OUTPUT_DIR, "face_landmarker.task")

# Camera / state
CAMERA = "FullHDwebcam"
STATES = ["before", "after"]
STATE_NAMES = {"before" : "rest", "after" : "post_exercise"}

# FPS 
FPS_FRONTAL = camera_fps["FullHDwebcam"]
FPS_FALLBACK = 29.90

# Signal processing
RPPG_BP_LOW, RPPG_BP_HIGH= 0.7, 3.5
BUTTER_ORDER = 4

# ECG 
ECG_FS = ecg_fs_actual # 500 Hz
ECG_DURATION_S = ecg_duration # ~30s
ECG_BP_LOW, ECG_BP_HIGH = 0.5, 40.0
ECG_LEAD = "II" # best QRS morphology for Pan-Tompkins

# PPG
PPG_FS = round(ppg_fs_actual) # ~100 Hz
PPG_BP_LOW, PPG_BP_HIGH = 0.5, 8.0

# SQI 

# Per-algorithm weight floor: prevents division-by-zero if all SQI near zero.
RPPG_SQI_MIN_FLOOR = 0.01

# Ensemble gate: much more lenient thatn NB4 0.4 autocorrelation gate
RPPG_SQI_THRESHOLD = 0.07

# ECG gate 
ECG_SQI_THRESHOLD = 0.5

# Face detection - skip recording if >20% frames have no detected face
NO_FACE_THRESHOLD = 20.0

# HR physiological limits
HR_MIN = 40
HR_MAX = 200

# Ensemble algorithms
ALGORITHMS = ["pos", "chrom", "green"]

# HDF5
HDF5_SKIP_KEY = "rppg_ensemble"

# Biomarker tiers
BIOMARKER_TIERS = {
    "respiratory_rate" : "tier_1_confident",
    "systolic_bp" : "tier_2_experimental",
    "diastolic_bp" : "tier_2_experimental",
    "spo2" : "tier_2_experimental",
    "arterial_stiffness" : "tier_2_experimental",
    "stress_psm25" : "tier_2_experimental",
    "age" : "tier_3_conditioning",
    "sex" : "tier_3_conditioning",
    "bmi" : "tier_3_conditioning",
    "temperature" : "tier_4_not_feasible",
    "glycated_hb" : "tier_4_not_feasible",
    "cholesterol" : "tier_4_not_feasible",
    "hemoglobin" : "tier_4_not_feasible",
}

BIOMARKER_RANGES = {
    "systolic_bp" : (70, 220),
    "diastolic_bp" : (40, 130),
    "spo2" : (85, 100),
    "respiratory_rate" : (6, 40),
    "arterial_stiffness" : (3, 18),
    "stress_psm25" : (0, 10),
    "age" : (18, 90),
    "bmi" : (16, 60),
    "temperature" : (35, 40),
    "glycated_hb" : (4, 15),
    "cholesterol" : (2, 12),
    "hemoglobin" : (7, 20),
}

# db.csv column mapping 
COLUMN_MAP = {
    "patient_id" : "subject_id",
    "step" : "state",
    "upper_ap" : "systolic_bp",
    "lower_ap" : "diastolic_bp",
    "saturation" : "spo2",
    "glycated_hemoglobin" : "glycated_hb",
    "respiratory" : "respiratory_rate",
    "rigidity" : "arterial_stiffness",
    "stress" : "stress_psm25",
}

# ROI landmark indices (MediaPipe face mesh)
FOREHEAD_LM    = [10, 338, 297, 332, 284, 251, 389, 356, 454,
                  323, 361, 288, 397, 365, 379, 378, 400, 377,
                  152, 148, 176, 149, 150, 136, 172, 58,  132,
                  93,  234, 127, 162, 21,  54,  103, 67,  109]
LEFT_CHEEK_LM  = [234, 227, 116, 123, 147, 213, 192, 214, 210,
                  211, 206, 203, 36,  101, 119, 229, 228]
RIGHT_CHEEK_LM = [454, 447, 345, 352, 376, 433, 416, 434, 430,
                  431, 426, 423, 266, 330, 348, 449, 448]
ROI_CONFIGS = {
    "forehead"    : FOREHEAD_LM,
    "left_cheek"  : LEFT_CHEEK_LM,
    "right_cheek" : RIGHT_CHEEK_LM,
}

print("NB_P2_06 configuration loaded.")
print(f"Output HDF5 : {HDF5_PATH}")
print(f"Camera : {CAMERA} | States : {STATES}")
print(f"FPS (frontal) : {FPS_FRONTAL:.4f} Hz (fallback: {FPS_FALLBACK})")
print(f"rPPG band : {RPPG_BP_LOW}-{RPPG_BP_HIGH} Hz")
print(f"Algorithms : {ALGORITHMS}")
print(f"Ensemble SQI ≥ : {RPPG_SQI_THRESHOLD} (NB_P2_04 used 0.4 on POS alone)")
print(f"ECG SQI ≥ : {ECG_SQI_THRESHOLD}")
print(f"Skip key : `{HDF5_SKIP_KEY}` dataset present = done")

NB_P2_06 configuration loaded.
Output HDF5 : E:/QualityPhys\mcd_rppg_ensemble.h5
Camera : FullHDwebcam | States : ['before', 'after']
FPS (frontal) : 29.9000 Hz (fallback: 29.9)
rPPG band : 0.7-3.5 Hz
Algorithms : ['pos', 'chrom', 'green']
Ensemble SQI ≥ : 0.07 (NB_P2_04 used 0.4 on POS alone)
ECG SQI ≥ : 0.5
Skip key : `rppg_ensemble` dataset present = done


## 5. Build Recording Inventory

In [ ]:
def build_inventory(mcd_dir: str, camera: str, states: list) -> pd.DataFrame:
    """
    Build per-recording inventory

    Single camera (FullHDwebcam frontal) x 2 states = 1200 rows.
    """
    db_path = os.path.join(mcd_dir, "db.csv")
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"db.csv not found at {db_path}")

    df_db = pd.read_csv(db_path)
    df_cam = df_db[(df_db["camera"] == camera) & (df_db["step"].isin(states))].copy().reset_index(drop=True)
    df_cam = df_cam.rename(columns={key: value for key, value in COLUMN_MAP.items() if key in df_cam.columns})
    print(f"Rows after filter : {len(df_cam)} (expected {600 * len(states)})")
    
    # Resolve video paths
    records = []
    for _, row, in df_cam.iterrows():
        subject_id = int(row["subject_id"])
        state = row["state"]
        video_path = os.path.join(mcd_dir, row["video"])
        ecg_path = os.path.join(mcd_dir, row["ecg"]) if pd.notna(row.get("ecg")) else None
        ppg_path = os.path.join(mcd_dir, row["ppg"]) if pd.notna(row.get("ppg")) else None
        has_video = os.path.exists(video_path)

        # FPS per video header
        fps = FPS_FALLBACK
        if has_video:
            cap = cv2.VideoCapture(video_path)
            fps_header = cap.get(cv2.CAP_PROP_FPS)
            cap.release()
            fps = fps_header if fps_header > 0 else FPS_FALLBACK

        # Biomarkers
        biomarkers = {}
        for bm in BIOMARKER_TIERS:
            raw = row.get(bm, np.nan)
            try:
                biomarkers[bm] = float(raw)
            except (ValueError, TypeError):
                biomarkers[bm] = raw # "F"/"M" 
                

        record = {
            "subject_id" : subject_id,
            "state" : state,
            "camera" : camera,
            "video_path" : video_path if has_video else None,
            "ecg_path" : ecg_path if ecg_path and os.path.exists(ecg_path) else None,
            "ppg_path" : ppg_path if ppg_path and os.path.exists(ppg_path) else None,
            "fps" : fps,
            "has_video" : has_video,
        }
        record.update(biomarkers)
        records.append(record)
        
    df = pd.DataFrame(records).sort_values(["subject_id", "state"]).reset_index(drop=True)

    # HDF5 completion flag
    # Mark rows already processed so the main loop can skip them.
    df["done"] = False
    if os.path.exists(HDF5_PATH):
        with h5py.File(HDF5_PATH, "r") as hf:
            for i, row in df.iterrows():
                hdf5_key = f"subjects/{row['subject_id']:04d}/recordings/{row['state']}/{HDF5_SKIP_KEY}"
                if hdf5_key in hf:
                    df.at[i, "done"] = True
    return df

# Run 
df_inventory = build_inventory(MCD_DIR, CAMERA, STATES)

print(f"Total recordings  {len(df_inventory)}")
print(f"Videos found : {df_inventory['has_video'].sum()}")
print(f"With ECG : {df_inventory['ecg_path'].notna().sum()}")
print(f"With PPG : {df_inventory['ppg_path'].notna().sum()}")
print(f"Already done : {df_inventory['done'].sum()}")
print(f"Remaining : {(df_inventory['has_video'] & ~df_inventory['done']).sum()}")
print()

fps_counts = df_inventory["fps"].value_counts().sort_index()
print("FPS distribution:")
for fps_val, count in fps_counts.items():
    flag = " <- fallback" if fps_val == FPS_FALLBACK else ""
    print(f"{fps_val:.4f} Hz  {count} videos{flag}")
print()

print("Per state:")
for state in STATES:
    sub = df_inventory[df_inventory["state"] == state]
    done = sub["done"].sum()
    total = sub["has_video"].sum()
    print(f"{state:<8} : videos={total} done={done} remaining={total - done}")

Rows after filter : 1200 (expected 1200)


## 6. Reference Signal Loaders

In [ ]:
def bandpass_filter(signal: np.ndarray, fs: float, low: float, high: float, order: int = 4) -> np.ndarray:
    """Zero-phase Butterworth bandpass."""
    nyq = fs / 2.0
    b,a = butter(order, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, signal).astype(np.float32)


def zscore(signal: np.ndarray) -> np.ndarray:
    """Zero-mean unit variance normalisation."""
    mu, sigma = signal.mean(), signal.std() + 1e-8
    return ((signal - mu) / sigma).astype(np.float32)


def resample_to_fps(signal: np.ndarray, original_fs: float, target_fps: float) -> np.ndarray:
    """Resample to video FPS via linear interpolation."""
    n_orig = len(signal)
    n_target = int(n_orig * target_fps / original_fs)
    return np.interp(np.linspace(0, 1, n_target), np.linspace(0, 1, n_orig), signal).astype(np.float32)


def load_ecg_signal(ecg_path: str) -> dict | None:
    """
    Load MCD-rPPG ECG from JSON.
    Confirmed structure: frequency=500 Hz, data=[{title, values}].
    Lead II used for Pan-Tompkins R-peak detection.
    """
    if not ecg_path or not os.path.exists(ecg_path):
        return None
    try: 
        with open(ecg_path) as file:
            ecg_json = json.load(file)
    except (json.JSONDecodeError, OSError) as error:
        print(f"WARNING: corrupted ECG JSON - {error}")
        return None

    ecg_fs = float(ecg_json.get("frequency", ECG_FS))
    leads = {lead["title"]: np.array(lead["values"], dtype=np.float32) for lead in ecg_json["data"]}

    if "II" in leads:
        ecg_raw, lead_used = leads["II"], "II"
    elif "I" in leads:
        ecg_raw, lead_used = leads["I"], "I"
    else:
        lead_used = list(leads.keys())[0]
        ecg_raw = leads[lead_used]
        print(f"WARNING: lead II/I not found, using {lead_used}")

    return {
        "ecg_raw" : ecg_raw,
        "timestamps" : np.arange(len(ecg_raw)) / ecg_fs,
        "ecg_fs" : ecg_fs,
        "lead" : lead_used,
    }


def load_ppg_signal(ppg_path: str) -> dict | None:
    """
    Load MCD-rPPG PPG from .PW format
    Confirmed structure: int_amplitude YYYY-MM-DD HH:MM:SS.ffffff
    fs estimated from inter-sample timestamps.
    """
    if not ppg_path or not os.path.exists(ppg_path):
        return None

    try:
        df = pd.read_csv(ppg_path, sep=r"\s+", header=None, dtype=str)
        ppg_raw = df.iloc[:, 0].astype(np.float32).values
        ts_str = df.iloc[:, 1] + " " + df.iloc[:, 2]
        ts_dt = pd.to_datetime(ts_str, format="mixed")
        ts_s = (ts_dt - ts_dt.iloc[0]).dt.total_seconds().values.astype(np.float64)
    except Exception as error:
        print(f"WARNING: corrupted PPG file - {error}")
        return None
    dt = np.diff(ts_s).mean() if len(ts_s) > 1 else 1.0 / PPG_FS
    ppg_fs = 1.0 / dt if dt > 0 else float(PPG_FS)

    return {"ppg_raw" : ppg_raw, "timestamps" : ts_s, "ppg_fs" : ppg_fs}


def detect_ecg_peaks(ecg_raw: np.ndarray, ecg_fs: float) -> tuple:
    """
    Bandpass -> nk.ecg_clean -> nk.ecg_peaks.
    Returns (peaks, rr_ms, hr_mean, ecg_filt).
    """
    ecg_filt = bandpass_filter(ecg_raw, ecg_fs, ECG_BP_LOW, ECG_BP_HIGH)
    ecg_clean = nk.ecg_clean(ecg_filt, sampling_rate=int(ecg_fs))
    peak_dict, _ = nk.ecg_peaks(ecg_clean, sampling_rate=int(ecg_fs))
    peak_idx = np.where(peak_dict["ECG_R_Peaks"] == 1)[0]

    if len(peak_idx) < 3:
        return peak_idx, np.array([]), float("nan"), ecg_filt

    rr_ms = np.diff(peak_idx) / ecg_fs * 1000.0
    hr_mean = float(60000.0 / rr_ms.mean())
    return peak_idx, rr_ms, hr_mean, ecg_filt


def detect_ppg_peaks(ppg_raw: np.ndarray, ppg_fs: float) -> tuple:
    """
    Bandpass -> nk.ppg_clean -> nk.ppg_peaks.
    Returns (peaks, rr_ms, hr_mean).
    """
    ppg_filt = bandpass_filter(ppg_raw, ppg_fs, PPG_BP_LOW, PPG_BP_HIGH)
    ppg_clean = nk.ppg_clean(ppg_filt, sampling_rate=int(ppg_fs))
    peak_dict, _ = nk.ppg_peaks(ppg_clean, sampling_rate=int(ppg_fs))
    peak_idx = np.where(peak_dict["PPG_Peaks"] == 1)[0]

    if len(peak_idx) < 3:
        return peak_idx, np.array([]), float("nan")

    rr_ms = np.diff(peak_idx) / ppg_fs * 1000.0
    hr_mean = float(60000.0 / rr_ms.mean())
    return peak_idx, rr_ms, hr_mean


def compute_hrv_features(rr_ms: np.ndarray) -> dict:
    """Time-domain HRV from RR intervals."""
    nan = {
        "mean_rr_ms" : float("nan"),
        "sdnn_ms" : float("nan"),
        "rmssd_ms" : float("nan"), 
        "pnn50_pct" : float("nan"),
        "hr_mean_bpm" : float("nan")
    }

    if len(rr_ms) < 3:
        return nan
    
    rr = rr_ms[(rr_ms > 300) & (rr_ms < 2000)]
    if len(rr) < 3:
        return nan
    diff = np.diff(rr)
    return  {
        "mean_rr_ms" : float(rr.mean()),
        "sdnn_ms" : float(rr.std()),
        "rmssd_ms" : float(np.sqrt(np.mean(diff ** 2))), 
        "pnn50_pct" : float((np.abs(diff) > 50).mean() * 100),
        "hr_mean_bpm" : float(60000.0 / rr.mean())
    }

    
def compute_ecg_sqi(rr_ms: np.ndarray) -> float:
    """
    ECG SQI: 1 - CV(RR).
    CV = std / mean
    Range [0, 1]
    """
    if len(rr_ms) < 3:
        return float("nan")
    rr = rr_ms[(rr_ms > 300) & (rr_ms < 2000)]
    if len(rr) < 3:
        return float("nan")
    cv = rr.std() / (rr.mean() + 1e-8)
    return float(1.0 - min(cv, 1.0))


def build_hr_continuous(peak_idx: np.ndarray, rr_ms: np.ndarray, source_fs: float, n_frames: int, target_fps: float) -> np.ndarray:
    """Continuous HR array at video frame rate, forward-filled from PPG peaks."""
    hr_cont = np.full(n_frames, np.nan, dtype=np.float32)
    if len(peak_idx) < 2 or len(rr_ms) == 0:
        return hr_cont
    for i, idx in enumerate(peak_idx[:-1]):
        fi = int(idx / source_fs * target_fps)
        if 0 <= fi < n_frames:
            hr_cont[fi] = 60000.0 / rr_ms[i]
    last = np.nan
    for i in range(n_frames):
        if not np.isnan(hr_cont[i]):
            last = hr_cont[i]
        elif not np.isnan(last):
            hr_cont[i] = last
    return hr_cont

print("Reference signal fucntions defined.")

## 7. MediaPipe ROI Extraction


Same Tasks API + face_landmarker.task as all prior CRVSE notebooks.
Single-pass extraction returns `roi_rgb [T, n_rois, 3]` - shared input to all three algorithms.

In [ ]:
# Download MediaPipe face landmarker model if not present
if not os.path.exists(MP_MODEL_PATH):
    print("Downloading MediaPipe face landmarker model...")
    url = ("https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task")
    urllib.request.urlretrieve(url, MP_MODEL_PATH)
    print("Downloaded.")
else:
    print(f"Face landmarker model present: {MP_MODEL_PATH}")


def get_roi_mask(landmarks, frame_shape: tuple, lm_indices: list) -> np.ndarray:
    """Binary polygon mask for one ROI from MediaPipe landmarks."""
    h, w = frame_shape[:2]
    pts = np.array([[int(landmarks[i].x * w), int(landmarks[i].y * h)] for i in lm_indices], dtype=np.int32)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [pts], 1)
    return mask


def extract_roi_signals(video_path: str, roi_configs: dict, max_frames: int = None) -> dict:
    """
    Single-pass video extraction.
    Returns:
        roi_rgb : [T, n_rois, 3] spatially-averaged RGB per ROI
        actual_fps : float = read from this specific video header
        n_frames : int 
        no_face_pct : float - % frames without face detection
    """
    cap = cv2.VideoCapture(video_path)
    actual_fps = cap.get(cv2.CAP_PROP_FPS)
    if actual_fps <= 0:
        actual_fps = FPS_FALLBACK

    roi_names = list(roi_configs.keys())
    roi_sigs = {name : [] for name in roi_names}
    n_frames = 0
    no_face = 0

    base_opts = mp_python.BaseOptions(model_asset_path=MP_MODEL_PATH)
    options = mp_vision.FaceLandmarkerOptions(
        base_options=base_opts,
        running_mode=mp_vision.RunningMode.VIDEO,
        num_faces=1,
    )

    with mp_vision.FaceLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if max_frames and n_frames >= max_frames:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            ts_ms = int((n_frames / actual_fps) * 1000)
            result = landmarker.detect_for_video(mp_img, ts_ms)

            if result.face_landmarks:
                lm = result.face_landmarks[0]
                for name, indices in roi_configs.items():
                    mask = get_roi_mask(lm, rgb.shape, indices)
                    if mask.sum() > 0: 
                        r = rgb[:, :, 0][mask == 1].mean()
                        g = rgb[:, :, 1][mask == 1].mean()
                        b = rgb[:, :, 2][mask == 1].mean()
                        roi_sigs[name].append(np.array([r, g, b], dtype=np.float32))
                    else:
                        fb = roi_sigs[name][-1] if roi_sigs[name] else np.zeros(3, np.float32)
                        roi_sigs[name].append(fb)
            else:
                no_face += 1
                for name in roi_configs:
                    fb = roi_sigs[name][-1] if roi_sigs[name] else np.zeros(3, np.float32)
                    roi_sigs[name].append(fb)

            n_frames += 1

    cap.release()

    # Stack to [T, n_rois, 3]
    arrays = [np.array(roi_sigs[name]) for name in roi_names]
    roi_rgb = np.stack(arrays, axis=1).astype(np.float32) # [T, n_rois, 3]

    return {
        "roi_rgb" : roi_rgb,
        "actual_fps" : actual_fps,
        "n_frames" : n_frames,
        "no_face_pct" : round(100 * no_face / max(n_frames, 1), 1),
    }

print("ROI extraction function defined.")

## 8. rPPG Algorithms + Spectral SQI + Ensemble

Three algorithms applied to the same `roi_rgb [T, n_rois, 3]` array.
SQI computed independently per algorithm - no reference HR required.
Ensemble = quality-weighted sum of z-scored signals.

In [ ]:
def apply_pos(roi_rgb: np.ndarray, fps: float) -> np.ndarray:
    """
    POS algorithm
    Projects normalised chrominance onto plane orthogonal to skin-tone vector.

    For each ROI independently, then average:
        Cn = C / mean(C) (temporal normalisation)
        S = P @ Cn.T (project to orthogonal plane)
        h = S[0] + alpha * S[1] (combine with adaptive alpha)
    """
    P = np.array([[0, 1, -1], [-2, 1, 1]], dtype=np.float64)
    n_rois = roi_rgb.shape[1]
    signals = []

    for r in range(n_rois):
        C = roi_rgb[:, r, :].astype(np.float64) # [T.3]
        Cn = C / (C.mean(axis=0) + 1e-8) # temporal norm
        S = P @ Cn.T # [2, T]
        h = S[0] + (S[0].std() / (S[1].std() + 1e-8)) * S[1]
        signals.append(h)

    combined = np.mean(signals, axis=0)
    filtered = bandpass_filter(combined, fps, RPPG_BP_LOW, RPPG_BP_HIGH)
    return zscore(filtered)


def apply_chrom(roi_rgb: np.ndarray, fps: float) -> np.ndarray:
    """
    CHROM algorithm
    Models skin reflectance as linear mixture of specular + diffuse components.

    For each ROI:
        Cn = C / mean(C) (temporal normalisation)
        Xs = 3*R - 2*G (chrominance 1)
        Ys = 1.5*R + G - 1.5*B (chrominance 2)
        H = Xs - alpha * Ys (alpha = std(xs)/std(ys) cancels specular)

    Strenghts: specular glare rejection.
    Known limitation: documented MAE degradation on darker skin tones.
    """
    n_rois = roi_rgb.shape[1]
    signals = []

    for r in range(n_rois):
        C = roi_rgb[:, r, :].astype(np.float64) # [T, 3]
        Cn = C / (C.mean(axis=0) + 1e-8)
        Xs = 3.0 * Cn[:,0] - 2.0 * Cn[:, 1]
        Ys = 1.5 * Cn[:, 0] + Cn[:, 1] - 1.5 * Cn[:, 2]
        alpha = Xs.std() / (Ys.std() + 1e-8)
        signals.append(Xs - alpha * Ys)

    combined = np.mean(signals, axis=0)
    filtered = bandpass_filter(combined, fps, RPPG_BP_LOW, RPPG_BP_HIGH)
    return zscore(filtered)


def apply_green(roi_rgb: np.ndarray, fps: float) -> np.ndarray:
    """
    GREEN channel mean across ROIs.
    Physiological basis: oxyhaemoglobin absorption peak ~550 nm (green).
    No artefact rejection - interpretable performance floor / baseline.
    """
    green = roi_rgb[:, :, 1].mean(axis=1) # [T] - mean green across ROIs
    filtered = bandpass_filter(green, fps, RPPG_BP_LOW, RPPG_BP_HIGH)
    return zscore(filtered)


# Algorithm dispatch
ALGORITHM_FN = {
    "pos" : apply_pos,
    "chrom" : apply_chrom,
    "green" : apply_green
}


def compute_spectral_sqi(signal: np.ndarray, fps: float) -> float:
    """
    Peak-to-band power ratio in cardiac band [0.7-3.5 Hz].

    SQI = P_dominant_peak / P_band_total

    Interpretation: 
        A periodic cardiac signal concentrates power at one dominant frequency.
        SQI -> 1.0: near-sinusoidal (very clean).
        SQI -> 0.1: power spread across the cardiac band (noisy but detectable).
        SQI -> 0.0: no dominant peak (noise).

    """
    N = len(signal)
    freqs = np.fft.rfftfreq(N, 1.0 / fps)
    power = np.abs(np.fft.rfft(signal - signal.mean())) ** 2

    cardiac_mask = (freqs >= RPPG_BP_LOW) & (freqs <= RPPG_BP_HIGH)
    band_power = power[cardiac_mask]

    if band_power.sum() < 1e-10:
        return 0.0

    peak_idx = np.argmax(band_power)
    low = max(0, peak_idx - 3)
    high = min(len(band_power), peak_idx + 4)
    peak_power = band_power[low:high].sum()

    return float(peak_power / band_power.sum())


def compute_ensemble(signals: dict, sqis: dict) -> tuple:
    """
    Quality-proportional weighted sum of z-scored rPPG signals.

    w_i = max(SQI_i, RPPG_SQI_MIN_FLOOR) / sum(w_j)

    The floor prevents division-by-zero when all SQI values are near zero.
    A floored weight means even a very weak algorithm contributes minimally
    rather than being hard-excluded.

    Returns: 
        ensemble : np.ndarray [T]
        weights : np.ndarray [n_alorithms] (sums ot 1)
    """
    raw_weights = np.array([max(sqis[alg], RPPG_SQI_MIN_FLOOR) for alg in ALGORITHMS], dtype=np.float64)
    weights = raw_weights / raw_weights.sum()

    stacked = np.stack([signals[alg] for alg in ALGORITHMS], axis=0) # [3, T]
    ensemble = (stacked * weights[:, None]).sum(axis=0).astype(np.float32)

    return ensemble, weights.astype(np.float32)


print("RPPG algorithm + SQI + ensemble functions defined.")
print(f"Algorithms : {ALGORITHMS}")
print("SQI type : spectral peak-to-band ratio (no reference HR required)")
print("Ensemble : quality-weighted soft fusion")

## 9. Single-Recording Pipeline 

Skip conditions: 

* `no_ecg_no_ppg` - no reference signals
* `ecg_too_short` - ECG < 10 s
* `flat_ecg` / `flat_ppg` - sensor issue
* `ecg_sqi_low` - RR irregularity (SQI < 0.5)
* `no_ppg` - PPG missing (needed for hr_continuous)
* `implausible_hr` - PPG HR outside 40-200 BPM
* `ecg_ppg_hr_mismatch` - ECG vs PPG HR differ > 10 BPM
* `no_face` -> 20% frames without face detection
* `ensemble_sqi_low` - ensemble spectral SQI < 0.07

In [ ]:
def process_single_recording(row: pd.Series, max_frames: int = None) -> tuple:
    """
    Full ensemble rPPG preprocessing pipeline for one MCD-rPPG recording.

    Reference strategy:
        ECG (~30 s, 500 Hz) -> HRV features
        PPG (~180s, ~100 Hz) -> hr_continuous + reference_signal

    rPPG strategy:
        roi_rgb extracted once -> POS + CHROM + GREEN applied in parallel
        Per-algorithm spectral SQI -> quality-weighted ensemble
        Gate on ENSEMBLE SQI

    Returns:
        (result_dict, reason_string)
        result_dict is None if recording is skipped.
    """
    subject_id = int(row["subject_id"])
    state = row["state"]
    label = f"s{subject_id:04d}/{state}"

    # Step 1: Reference signals
    ecg_data = load_ecg_signal(row["ecg_path"]) if pd.notna(row.get("ecg_path")) else None
    ppg_data = load_ppg_signal(row["ppg_path"]) if pd.notna(row.get("ppg_path")) else None

    if ecg_data is None and ppg_data is None:
        return None, "no_ecg_no_ppg"

    # Step 2: ECG processing (HRV - 30 s window)
    ecg_sqi = float("nan")
    rr_ms = np.array([])
    hr_mean_ecg = float("nan")

    if ecg_data is not None:
        ecg_raw = ecg_data["ecg_raw"]
        ecg_fs = ecg_data["ecg_fs"]

        if len(ecg_raw) / ecg_fs < 10.0:
            return None, f"ecg_too_short ({len(ecg_raw)/ecg_fs:.1f} s)"

        if ecg_raw.max() - ecg_raw.min() < 0.05:
            return None, "flat_ecg"

        _, rr_ms, hr_mean_ecg, _ = detect_ecg_peaks(ecg_raw, ecg_fs)

        if not np.isnan(hr_mean_ecg):
            ecg_sqi = compute_ecg_sqi(rr_ms)
            if ecg_sqi < ECG_SQI_THRESHOLD:
                return None, f"ecg_sqi_low ({ecg_sqi:.3f})"

    hrv = compute_hrv_features(rr_ms)

    # Step 3: PPG processing 
    if ppg_data is None:
        return None, "no_ppg"

    ppg_raw = ppg_data["ppg_raw"]
    ppg_fs = ppg_data["ppg_fs"]

    if ppg_raw.max() - ppg_raw.min() < 1.0:
        return None, "flat_ppg"

    ppg_peak_idx, rr_ms_ppg, hr_mean_ppg = detect_ppg_peaks(ppg_raw, ppg_fs)

    if np.isnan(hr_mean_ppg) or hr_mean_ppg < HR_MIN or hr_mean_ppg > HR_MAX:
        return None, f"implausible_hr ({hr_mean_ppg:.1f} BPM)"

    if not np.isnan(hr_mean_ecg) and not np.isnan(hr_mean_ppg):
        if abs(hr_mean_ecg - hr_mean_ppg) > (15.0 if state == "before" else 20.0):
            return None, (f"ecg_ppg_hr_mismatch (ECG={hr_mean_ecg:.1f} PPG={hr_mean_ppg:.1f})")

   

    # Step 4: ROI extraction from video
    roi_out = extract_roi_signals(row["video_path"], ROI_CONFIGS, max_frames=max_frames)
    roi_rgb = roi_out["roi_rgb"]
    n_frames = roi_out["n_frames"]
    actual_fps = roi_out["actual_fps"]
    no_face = roi_out["no_face_pct"]
    hr_mean = hr_mean_ppg
    ppg_resampled = resample_to_fps(ppg_raw, ppg_fs, actual_fps)
    if no_face > NO_FACE_THRESHOLD:
        return None, f"no_face ({no_face:.1f}%)"

    # Step 5: Three rPPG algorithms
    rppg_signals = {}
    rppg_sqis = {}
    for alg in ALGORITHMS:
        sig = ALGORITHM_FN[alg](roi_rgb, actual_fps)
        rppg_signals[alg] = sig
        rppg_sqis[alg] = compute_spectral_sqi(sig, actual_fps)

    # Step 6: Quality-weighted ensemble 
    rppg_ensemble, ens_weights = compute_ensemble(rppg_signals, rppg_sqis)
    sqi_ensemble = compute_spectral_sqi(rppg_ensemble, actual_fps)

    if sqi_ensemble < RPPG_SQI_THRESHOLD:
        return None, f"ensemble_sqi_low ({sqi_ensemble:.3f})"

    # Step 7: Build continuous HR from PPG
    hr_continuous = build_hr_continuous(ppg_peak_idx, rr_ms_ppg, ppg_fs, n_frames, actual_fps)

    # Step 8: Align all signals to shortest common length
    T = min(n_frames, len(rppg_ensemble), len(hr_continuous), len(ppg_resampled), roi_rgb.shape[0])

    rppg_pos = rppg_signals["pos"][:T]
    rppg_chrom = rppg_signals["chrom"][:T]
    rppg_green = rppg_signals["green"][:T]
    rppg_ensemble = rppg_ensemble[:T]
    roi_rgb = roi_rgb[:T]
    hr_continuous = hr_continuous[:T]
    reference_sig = ppg_resampled[:T]

    # Step 9: Biomarkers
    biomarkers = {}
    bm_flags = {}
    for bm in BIOMARKER_TIERS:
        raw = row.get(bm, np.nan)
        try:
            value = float(raw)
        except (ValueError, TypeError):
            value = raw
        biomarkers[bm] = value
        if bm in BIOMARKER_RANGES and isinstance(value, float) and not np.isnan(value):
            low, high = BIOMARKER_RANGES[bm]
            bm_flags[f"{bm}_valid"] = (low <= value <= high)
        else:
            bm_flags[f"{bm}_valid"] = False

    rr_for_storage = (rr_ms.astype(np.float32) if len(rr_ms) > 0 else rr_ms_ppg.astype(np.float32))
    sqis_str = " ".join(f"{a}={rppg_sqis[a]:.3f}" for a in ALGORITHMS)
    print(f"{label} T={T} HR={hr_mean:.1f} ECG_SQI={ecg_sqi:.2f} ens_SQI={sqi_ensemble:.3f} [{sqis_str}] no_face={no_face:.1f}% fps={actual_fps:.2f}")

    result = {
        # Ensemble and individual rPPG signals
        "rppg_pos" : rppg_pos,
        "rppg_chrom" : rppg_chrom,
        "rppg_green" : rppg_green,
        "rppg_ensemble" : rppg_ensemble,
        "ensemble_weights" : ens_weights, # [w_pos, w_chrom, w_green]
        # Supporting signals
        "roi_rgb" : roi_rgb,
        "reference_signal" : reference_sig,
        "hr_continuous" : hr_continuous,
        "rr_intervals"  : rr_for_storage,
        # HRV (ECG-derived)
        "hr_mean"  : float(hr_mean),
        "rmssd_ms" : hrv["rmssd_ms"],
        "sdnn_ms" : hrv["sdnn_ms"],
        "pnn50_pct" : hrv["pnn50_pct"],
        # SQI
        "ecg_sqi" : float(ecg_sqi) if not np.isnan(ecg_sqi) else -1.0,
        "sqi_pos" : float(rppg_sqis["pos"]),
        "sqi_chrom"  : float(rppg_sqis["chrom"]),
        "sqi_green"  : float(rppg_sqis["green"]),
        "sqi_ensemble" : float(sqi_ensemble),
        # Metadata
        "subject_id" : subject_id,
        "activity_id" : state,
        "activity_name" : STATE_NAMES.get(state, state),
        "camera" : CAMERA,
        "dataset" : DATASET_NAME,
        "reference_type" : "ECG_hrv_PPG_continuous",
        "n_frames" : T,
        "fps" : actual_fps,
        "no_face_pct" : float(no_face),
        # Biomarkers
        **biomarkers,
        **bm_flags,
    }
    return result, "ok"
        
print("process_single_recording_defined.")

## 10. Single-Recording Demo

Test on 300 frames (~10s) to confirm pipeline runs end to end before full run.

In [ ]:
first = df_inventory[df_inventory["has_video"]].iloc[0]
print(f"Demo: s{int(first.subject_id):04d}/{first.state} fps={first.fps:.4f}")
print()

demo_result, demo_reason = process_single_recording(first, max_frames=300)

if demo_result is not None:
    print()
    print("Signals")
    for key in ["rppg_pos", "rppg_chrom", "rppg_green", "rppg_ensemble", "roi_rgb", 
                "reference_signal", "hr_continuous", "rr_intervals", "ensemble_weights"]:
        arr = demo_result.get(key)
        if arr is not None and hasattr(arr, "shape"):
            print(f"{key:<22} : {arr.shape} dtype={arr.dtype}")
    print()
    print("Scalars")
    for key in ["hr_mean", "rmssd_ms", "ecg_sqi", "sqi_pos", "sqi_chrom", "sqi_green", "sqi_ensemble", "reference_type", "fps"]:
        value = demo_result.get(key, "N/A")
        fmt = f"{value:.4f}" if isinstance(value, float) else str(value)
        print(f"{key:<22} : {fmt}")

    print()
    print("Ensemble weights")
    for alg, w in zip(ALGORITHMS, demo_result["ensemble_weights"]):
        print(f"{alg:<6} : {w:.4f}")
        

## 11. HDF5 Write Functions

Same schema as NB_P2_04 (`subjects/{subj}/recordings/{state}/`) with additional ensemble datasets and per-algorithm SQI attributes.

In [ ]:
def initialise_hdf5(output_path: str) -> h5py.File:
    """
    Open or create mcd_rppg_ensemble.h5 in append mode.
    Top-level metadata written once on first open.
    """
    hf = h5py.File(output_path, "a")
    if "metadata" not in hf:
        meta = hf.require_group("metadata")
        meta.attrs["dataset_name"] = DATASET_NAME
        meta.attrs["preprocessing_version"] = "1.0"
        meta.attrs["fps"] = FPS_FRONTAL
        meta.attrs["ecg_fs"] = ECG_FS
        meta.attrs["ecg_duration_s"] = ECG_DURATION_S
        meta.attrs["ppg_fs"] = PPG_FS
        meta.attrs["rppg_bp_low"] = RPPG_BP_LOW
        meta.attrs["rppg_bp_high"]= RPPG_BP_HIGH
        meta.attrs["ecg_bp_low"] = ECG_BP_LOW
        meta.attrs["ecg_bp_high"] = ECG_BP_HIGH
        meta.attrs["ppg_bp_low"] = PPG_BP_LOW
        meta.attrs["ppg_bp_high"] = PPG_BP_HIGH
        meta.attrs["rppg_sqi_threshold"] = RPPG_SQI_THRESHOLD
        meta.attrs["ecg_sqi_threshold"] = ECG_SQI_THRESHOLD
        meta.attrs["camera_selected"] = CAMERA
        meta.attrs["states"] = str(STATES)
        meta.attrs["algorithms"] = str(ALGORITHMS)
        meta.attrs["sqi_type"] = "spectral_peak_to_band"
        meta.attrs["n_biomarkers"] = len(BIOMARKER_TIERS)
        meta.attrs["biomarker_names"] = str(list(BIOMARKER_TIERS.keys()))
        meta.attrs["reference_type"] = "ECG_hrv_PPG_continuous"
        meta.attrs["note"] = (
            "Ensemble rPPG: quality-weighted POS+CHROM+GREEN. SQI gate on ensemble (0) vs NB_P2_04 POS-only gate (0.40). "
            "Expected higher pass rate than NB_P2_04 11%."
        )
    return hf


def write_recording_to_hdf5(hf: h5py.File, result: dict) -> bool:
    """
    Write one recording. Returns True if written, False if already exists.
    Path: subjects/{subj:04d}/recordings/{state}
    """
    subj = f"{result['subject_id']:04d}"
    path = f"subjects/{subj}/recordings/{result['activity_id']}"

    if path in hf:
        return False

    grp = hf.require_group(path)

    # Signal dataset
    for ds_name in ["rppg_pos", "rppg_chrom", "rppg_green", "rppg_ensemble", "roi_rgb", 
                    "reference_signal", "hr_continuous", "rr_intervals", "ensemble_weights"]:
        grp.create_dataset(ds_name, data=result[ds_name], compression="gzip", compression_opts=4)

    # Standard attributes
    std = {
        "subject_id" : result["subject_id"],
        "activity_id" : result["activity_id"],
        "activity_name" : result["activity_name"],
        "camera" : result["camera"],
        "dataset" : result["dataset"],
        "reference_type" : result["reference_type"],
        "n_frames" : result["n_frames"],
        "fps" : result["fps"],
        "hr_mean" : result["hr_mean"],
        "rmssd_ms" : result["rmssd_ms"] if not np.isnan(result["rmssd_ms"]) else -1.0,
        "sdnn_ms" : result["sdnn_ms"] if not np.isnan(result["sdnn_ms"]) else -1.0,
        "pnn50_pct" : result["pnn50_pct"] if not np.isnan(result["pnn50_pct"]) else -1.0,
        "ecg_sqi" : result["ecg_sqi"],
        "sqi_pos" : result["sqi_pos"],
        "sqi_chrom" : result["sqi_chrom"],
        "sqi_green" : result["sqi_green"],
        "sqi_ensemble" : result["sqi_ensemble"],
        "no_face_pct" : result["no_face_pct"],
    }
    for key, value in std.items():
        grp.attrs[key] = value

    # Biosignal attributes
    for bm, tier in BIOMARKER_TIERS.items():
        value = result.get(bm, np.nan)
        flag = result.get(f"{bm}_valid", False)
        if isinstance(value, str):
            grp.attrs[f"bm_{bm}"] = value
        elif value is None or (isinstance(value, float) and np.isnan(value)):
            grp.attrs[f"bm_{bm}"] = -999.0
        else:
            grp.attrs[f"bm_{bm}"] = float(value)
        grp.attrs[f"bm_{bm}_tier"]  = tier
        grp.attrs[f"bm_{bm}_valid"] = bool(flag)

    return True

# Demo write + readback
if demo_result is not None:
    print(f"Writing demo to: {HDF5_PATH}")
    with initialise_hdf5(HDF5_PATH) as hf:
        written = write_recording_to_hdf5(hf, demo_result)
    print(f"Written: {written}")
    # Readback
    with h5py.File(HDF5_PATH, "r") as hf:
        subj = f"{demo_result['subject_id']:04d}"
        grp = hf[f"subjects/{subj}/recordings/{demo_result['activity_id']}"]
        print()
        print("Datasets in HDF5 group:")
        for ds in grp.keys():
            print(f"{ds:<22}: {grp[ds].shape}")
        print()
        print("Key attributes:")
        for key in ["hr_mean", "sqi_pos", "sqi_chrom", "sqi_green", "sqi_ensemble", "ecg_sqi", "reference_type"]:
            print(f"{key:<22}: {grp.attrs.get(key, 'N/A')}")
        print()
        print("Ensemble weights stored:")
        print(f"{grp['ensemble_weights'][:]}")
else:
    print("No demo result to write.")

## 12. Process All Subjects

In [ ]:
def process_all_subjects(df_inventory: pd.DataFrame, hdf5_path: str, flush_every: int = 10) -> pd.DataFrame:
    """
    Full run with safe stop / resume / crash recorvery.
    Logs per-algorithm SQI + ensemble SQI + weights for post-hoc analysis.
    """
    STOP_FILE = os.path.join(OUTPUT_DIR, "STOP_PREPROCESSING.txt")
    LOG_PATH = os.path.join(OUTPUT_DIR, f"processing_log_{DATASET_NAME}.csv")

    log_rows = []
    existing_keys = set()
    stop_requested = False
    processed_count = 0

    # Resume: load existing log
    if os.path.exists(LOG_PATH) and os.path.getsize(LOG_PATH) > 0:
        df_prev = pd.read_csv(LOG_PATH)
        log_rows = df_prev.to_dict("records")
        existing_keys = set(zip(df_prev["subject_id"].astype(str), df_prev["state"]))
        print(f"Resuming - {len(log_rows)} entries from previous run.")
    else:
        print("Starting fresh run.")

    complete = df_inventory[df_inventory["has_video"]]
    total = len(complete)
    already = sum(1 for _, r in complete.iterrows() if (str(int(r["subject_id"])), r["state"]) in existing_keys)
    if already:
        print(f"Skipping {already}/{total} already logged.")
    print(f"To stop safely: create {STOP_FILE}")
    print()

    try:
        with initialise_hdf5(hdf5_path) as hf:
            
            for i, (_, row) in enumerate(complete.iterrows()):
                sid = int(row["subject_id"])
                state = row["state"]
                key = (str(sid), state)

                # Stop file
                if os.path.exists(STOP_FILE):
                    print(f"\nStop file detected at [{i+1}/{total}].")
                    stop_requested = True 
                    break

                # Log resume
                if key in existing_keys:
                    continue

                # HDF5 resume (crash recovery - log may be missing entry)
                hdf5_path_grp = f"subjects/{sid:04d}/recordings/{state}"
                if hdf5_path_grp in hf:
                    nan_fields  = {k: np.nan for k in ["hr_mean","ecg_sqi","sqi_pos", "sqi_chrom","sqi_green",
                                                       "sqi_ensemble", "n_frames","no_face_pct","fps"]}
                    log_rows.append({
                        "subject_id" : sid,
                        "state" : state,
                        "status" : "resumed",
                        "skip_reason" : "already_in_hdf5",
                        **nan_fields ,
                    })
                    existing_keys.add(key)
                    continue

                # Progress print every 50
                if (i + 1) % 50 == 0 or i == 0:
                    n_proc = sum(1 for r in log_rows if r.get("status") == "processed")
                    print(f"[{i+1:04d}/{total}]  {(i+1)/total*100:.1f}% processed this session: {processed_count} total in log: {n_proc}")

                print(f"[{i+1:04d}/{total}] s{sid:04d}/{state}", end="  ")
                result, reason = process_single_recording(row, max_frames=None)

                if result is not None:
                    write_recording_to_hdf5(hf, result)
                    processed_count += 1
                    log_entry = {
                        "subject_id" : sid,
                        "state" : state,
                        "camera" : CAMERA,
                        "status" : "processed",
                        "skip_reason" : "ok",
                        "hr_mean" : result["hr_mean"],
                        "ecg_sqi" : result["ecg_sqi"],
                        "sqi_pos" : result["sqi_pos"],
                        "sqi_chrom" : result["sqi_chrom"],
                        "sqi_green" : result["sqi_green"],
                        "sqi_ensemble" : result["sqi_ensemble"],
                        "w_pos" : float(result["ensemble_weights"][0]),
                        "w_chrom" : float(result["ensemble_weights"][1]),
                        "w_green" : float(result["ensemble_weights"][2]),
                        "n_frames" : result["n_frames"],
                        "no_face_pct" : result["no_face_pct"],
                        "fps" : result["fps"],
                    }
                    for bm in ["systolic_bp", "diastolic_bp", "spo2", "respiratory_rate", "age", "sex", "bmi"]:
                        log_entry[bm] = result.get(bm, np.nan)

                    log_rows.append(log_entry)
                    existing_keys.add(key)

                    if processed_count % flush_every == 0:
                        hf.flush()
                        pd.DataFrame(log_rows).to_csv(LOG_PATH, index=False)
                        print(f"[flush] HDF5 + log saved ({processed_count} new)")
                else:
                    print(f"SKIP: {reason}") 
                    nan_fields  = {k: np.nan for k in ["hr_mean","ecg_sqi","sqi_pos", "sqi_chrom","sqi_green", "sqi_ensemble", 
                                                    "w_pos","w_chrom","w_green", "n_frames","no_face_pct"]}
                    log_rows.append({
                        "subject_id" : sid,
                        "state" : state,
                        "camera" : CAMERA,
                        "status" : "skipped",
                        "skip_reason" : reason,
                        **nan_fields ,
                        "fps" : row["fps"],
                    })
                    existing_keys.add(key)
            
            hf.flush()
    except KeyboardInterrupt:
        print("\nKeyboardInterrupt - HDF5 closed safely.")

    # Final log save
    cols = ["subject_id","state","camera","status","skip_reason", "hr_mean","ecg_sqi","sqi_pos","sqi_chrom",
            "sqi_green","sqi_ensemble", "w_pos","w_chrom","w_green","n_frames","no_face_pct","fps"]

    df_log = pd.DataFrame(log_rows)
    for c in cols:
        if c not in df_log.columns:
            df_log[c] = np.nan
    df_log.to_csv(LOG_PATH, index=False)

    print()
    print("=" * 60)
    print("STOP - safe to shut down." if stop_requested else "Processing complete.")
    if stop_requested:
        print(f"Delete {STOP_FILE} before resuming.")
    print("=" * 60)

    n_proc = (df_log["status"] == "processed").sum()
    n_skip = (df_log["status"] == "skipped").sum()
    n_resumed = (df_log["status"] == "resumed").sum()

    print(f"Processed this session : {processed_count}")
    print(f"Total processed in log : {n_proc}")
    print(f"Skipped : {n_skip}")
    print(f"Resumed : {n_resumed}")
    print(f"Pass rate : {n_proc / (n_proc + n_skip) * 100:.1f}% (NB_P2_04 baseline: ~11%)")

    if n_skip > 0:
        print()
        print("Skip breakdown:")
        reasons = (df_log[df_log["status"] == "skipped"]["skip_reason"].str.split(" ").str[0].value_counts())
        for key, value in reasons.items():
            print(f"{key:<35} : {value}")

    if n_proc > 0:
        print()
        print("Per-algorithm SQI (processed recordings):")
        proc = df_log[df_log["status"] == "processed"]
        for alg in ALGORITHMS:
            col = f"sqi_{alg}"
            if col in proc.columns:
                s = proc[col].dropna()
                print(f"{alg:<8}: mean={s.mean():.3f} min={s.min():.3f} max={s.max():.3f}")
        if "sqi_ensemble" in proc.columns:
            s = proc["sqi_ensemble"].dropna()
            print(f"{'ensemble':<8}: mean={s.mean():.3f} min={s.min():.3f} max={s.max():.3f}")

    print(f"\nLog : {LOG_PATH}")
    print(f"HDF5 : {hdf5_path}")
    return df_log

df_log = process_all_subjects(df_inventory, HDF5_PATH)
print()
print(df_log[["subject_id", "state", "status", "skip_reason", "sqi_pos", "sqi_chrom", "sqi_green", "sqi_ensemble", "hr_mean"]].head(20).to_string())

## 13. Post-Procesing Validation


In [ ]:
def validate_hdf5(hdf5_path: str) -> pd.DataFrame:
    """
    Post-processing validation for mcd_rppg_ensemble.h5.

    Checks:
    1. Signal shape consistency 
    2. HR distribution by state
    3. Per-algorithm SQI distributions
    4. Ensemble weight distribution across algorithms
    5. Subject coverage 
    6. Pass rate vs NB_P2_04 baseline
    """
    records = []

    with h5py.File(hdf5_path, "r") as hf:
        print("=" * 60)
        print("Metadata:")
        for key, value in hf["metadata"].attrs.items():
            print(f"{key} : {value}")
        print()

        for subj_key in sorted(hf["subjects"].keys()):
            for act_key in sorted(hf[f"subjects/{subj_key}/recordings"].keys()):
                grp = hf[f"subjects/{subj_key}/recordings/{act_key}"]
                attrs = dict(grp.attrs)

                T = grp["rppg_ensemble"].shape[0]
                # Shape consistency checks
                assert grp["rppg_pos"].shape[0] == T, f"{subj_key}/{act_key}: rppg_pos lenght mismatch"
                assert grp["rppg_chrom"].shape[0] == T, f"{subj_key}/{act_key}: rppg_chrom length mismatch"
                assert grp["rppg_green"].shape[0] == T, f"{subj_key}/{act_key}: rppg_green length mismatch"
                assert grp["reference_signal"].shape[0] == T, f"{subj_key}/{act_key}:reference_signal mismatch"

                weights = grp["ensemble_weights"][:]

                rec = {
                    "subject_id" : attrs.get("subject_id"),
                    "state" : attrs.get("activity_id"),
                    "n_frames" : T,
                    "fps" : attrs.get("fps"),
                    "hr_mean" : attrs.get("hr_mean"),
                    "ecg_sqi" : attrs.get("ecg_sqi"),
                    "sqi_pos" : attrs.get("sqi_pos"),
                    "sqi_chrom" : attrs.get("sqi_chrom"),
                    "sqi_green" : attrs.get("sqi_green"),
                    "sqi_ensemble" : attrs.get("sqi_ensemble"),
                    "w_pos" : float(weights[0]),
                    "w_chrom" : float(weights[1]),
                    "w_green" : float(weights[2]),
                    "no_face_pct" : attrs.get("no_face_pct"),
                }
                for bm in BIOMARKER_TIERS:
                    rec[f"bm_{bm}"] = attrs.get(f"bm_{bm}", np.nan)
                records.append(rec)

    df = pd.DataFrame(records)

    print("=" * 60)
    print(f"Total recordings : {len(df)}")
    print(f"Unique subjects : {df['subject_id'].nunique()}")
    print(f"Pass rate : {len(df)/1200*100:.1f}% (NB_P2_04 baseline ~11%)")
    print()

    # HR by state
    print("HR by state (expect 'after' > 'before'):")
    print(df.groupby("state")["hr_mean"].agg(["mean", "std", "min", "max"]).round(1).to_string())
    print()

    # Per-algorithm SQI
    print("SQI distributions:")
    for col, label in [("sqi_pos","POS"), ("sqi_chrom","CHROM"), ("sqi_green","GREEN"), ("sqi_ensemble","ENSEMBLE")]:
        s = df[col].dropna()
        gate = RPPG_SQI_THRESHOLD if col =="sqi_ensemble" else None
        gate_str = f"(gate={gate})" if gate else ""
        print(f"{label:<10}: mean={s.mean():.3f} min={s.min():.3f} max={s.max():.3f}{gate_str}")
    print()

    # Ensemble weight distribution - which algorithm dominates?
    print("Ensemble weight distribution (mean across all recordings):")
    for alg, col in zip(ALGORITHMS, ["w_pos", "w_chrom", "w_green"]):
        w = df[col].dropna()
        print(f"{alg:<8}: mean={w.mean():.3f} std={w.std():.3f} min={w.min():.3f}  max={w.max():.3f}")
    print()

    # Subject coverage
    both = df.groupby("subject_id")["state"].nunique().eq(len(STATES)).sum()
    one = df["subject_id"].nunique() - both
    print(f"Subjects with both states : {both}")
    print(f"Subjects with 1 state : {one}")
    if one > 0:
        print(f"(expected if one state failed ECG or ensemble SQI gate")
    print()

    # State-variant biomarkers - exercise physiology sanity check
    state_bms = ["bm_systolic_bp", "bm_diastolic_bp", "bm_respiratory_rate"]
    state_cols = [c for c in state_bms if c in df.columns]
    if state_cols:
        print("State-variant biomarkers (expect 'after' > 'before' for BP/RR):")
        print(df.groupby("state")[state_cols].mean().round(1).to_string())
    print()

    # SQI comparision plot: per-algorithm + ensemble
    fig, axes = plt.subplots(1, 4, figsize=(16,4))
    fig.suptitle("SQI Distributions Ensemble", fontsize=11, fontweight="bold")
    plot_data = [("sqi_pos","POS","#74b9ff"), ("sqi_chrom","CHROM","#fd79a8"), ("sqi_green","GREEN","#55efc4"), ("sqi_ensemble","ENSEMBLE","#fdcb6e")]
    for ax, (col, label, color) in zip(axes, plot_data):
        s = df[col].dropna()
        ax.hist(s, bins=30, color=color, alpha=0.8)
        if col == "sqi_ensemble":
            ax.axvline(RPPG_SQI_THRESHOLD, color="#ff7675", ls="--", lw=1.5, label=f"{RPPG_SQI_THRESHOLD}")
            ax.legend(fontsize=8)
        ax.set_title(f"{label}\nmean={s.mean():.3f}", fontsize=9)
        ax.set_xlabel("SQI")
        ax.set_ylabel("Count")
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "nb06_sqi_distributions.png"), bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()

    return df
df_validation = validate_hdf5(HDF5_PATH)
    
                

In [ ]:
with h5py.File(HDF5_PATH, "a") as hf:
    hf["metadata"].attrs["note"] = (
        "Ensemble rPPG: quality-weighted POS+CHROM+GREEN. "
        "SQI gate on ensemble=0.07 (spectral peak-to-band, ±3 bins). "
        "Calibrated on 20-subject sample. NB_P2_04 POS-only gate was 0.40 "
        "autocorrelation — not directly comparable. "
        "Pass rate 42.2% vs NB_P2_04 11%."
    )
    print("Metadata note updated.")